# 02 — Insights via SQL

**Projeto:** Financial Behavior Intelligence  
**Objetivo:** Responder perguntas de negócio sobre o comportamento financeiro dos usuários utilizando queries SQL estruturadas via SQLite em memória.

---

## 0. Setup

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# carregar dataset tratado (output do notebook 01)
df = pd.read_csv('../data/processed/transactions_clean.csv')

# conexão SQLite em memória — portável, não depende de arquivo .db externo
conn = sqlite3.connect(':memory:')
df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.strftime('%Y-%m-%d')
df.to_sql('transacoes', conn, if_exists='replace', index=False)

# função auxiliar: executa qualquer query e retorna DataFrame
def query(sql):
    return pd.read_sql_query(sql, conn)

print(f'Banco SQLite em memória criado.')
print(f'Registros carregados: {len(df):,}')
print(f'Colunas: {df.columns.tolist()}')

Banco de dados criado com sucesso!
Tabela 'transacoes' alimentada com 19046 linhas.


Para garantir a robustez da análise de saúde financeira, os dados brutos em formato CSV foram processados via linguagem Python e armazenados em um banco de dados relacional SQLite. Este processo incluiu a normalização de tipos de dados e a padronização temporal, permitindo a execução de queries SQL complexas para extração de insights sobre padrões de consumo e margem de poupança mensal.

---
## Q1 — Variação de Despesas

##### Análise da saúde financeira ao longo do tempo. Esta query calcula o total de ganhos (créditos) e o total de gastos (débitos) para mostrar o saldo final.

> O usuário fecha o mês no azul ou no vermelho?

In [2]:
df_saldo_mensal = query("""
SELECT
    User_ID,
    strftime('%Y-%m', Date) as Mes,
    ROUND(SUM(CASE WHEN \"Transaction Type\" = 'credit' THEN Amount ELSE 0 END), 2) as Entradas,
    ROUND(SUM(CASE WHEN \"Transaction Type\" = 'debit'  THEN Amount ELSE 0 END), 2) as Saidas,
    ROUND(SUM(CASE WHEN \"Transaction Type\" = 'credit' THEN Amount ELSE 0 END) -
          SUM(CASE WHEN \"Transaction Type\" = 'debit'  THEN Amount ELSE 0 END), 2) as Saldo_Mensal
FROM transacoes
GROUP BY User_ID, Mes
ORDER BY User_ID, Mes;
""")

negativos = (df_saldo_mensal['Saldo_Mensal'] < 0).sum()
total     = len(df_saldo_mensal)
print(f'User-meses com saldo negativo: {negativos}/{total} ({negativos/total*100:.1f}%)')
print()
df_saldo_mensal.head(12)

        User_ID      Mes  Entradas    Saidas  Saldo_Mensal
0     USER_0001  2018-01   7162.89   2931.45       4231.44
1     USER_0001  2018-02   5220.75   3165.05       2055.70
2     USER_0001  2018-03   7321.50   3500.16       3821.34
3     USER_0001  2018-04   7166.88   6029.54       1137.34
4     USER_0001  2018-05   5091.55  11392.03      -6300.48
...         ...      ...       ...       ...           ...
1216  USER_0051  2024-08   3663.54   5233.88      -1570.34
1217  USER_0051  2024-09   7410.18   6640.26        769.92
1218  USER_0051  2024-10      0.00   8372.72      -8372.72
1219  USER_0051  2024-11   3734.33   3588.38        145.95
1220  USER_0051  2024-12   3664.01   3991.09       -327.08

[1221 rows x 5 columns]


Os dados revelam diferenças significativas entre usuários:
enquanto alguns mantêm saldo positivo consistente, outros acumulam déficits mês após mês.
A query Q5 irá quantificar exatamente quantos usuários fecham o mês no vermelho com frequência.


## Q2 — Ranking dos Gastos (Top 10 Categorias)

> Quais são os 10 maiores grupos de despesa?

In [3]:
df_top_gastos = query("""
SELECT
    Category as Categoria,
    ROUND(SUM(Amount), 2) as Gasto_Total
FROM transacoes
WHERE \"Transaction Type\" = 'debit'
GROUP BY Category
ORDER BY Gasto_Total DESC
LIMIT 10;
""")

fig, ax = plt.subplots(figsize=(10, 6))
df_top_gastos.sort_values('Gasto_Total').plot(
    kind='barh', x='Categoria', y='Gasto_Total',
    ax=ax, color='steelblue', legend=False
)
ax.set_title('Top 10 Categorias por Gasto Total (inclui movimentações financeiras)', fontsize=12)
ax.set_xlabel('Gasto Total (USD)')
plt.tight_layout()
plt.show()

print(df_top_gastos.to_string(index=False))

             Categoria  Gasto_Total
0                 Rent   1591300.84
1  Credit Card Payment    425432.19
2            Groceries    340873.59
3            Insurance    264164.37
4         Loan Payment    217126.37
5     Savings Transfer    215839.74
6            Utilities    193010.95
7           Gas & Fuel    161376.34
8               Travel    132858.62
9                Taxes    132280.45


O aluguel (Rent) é disparado a maior conta. Como insight, podemos verificar que o custo fixo de habitação é muito alto, a saúde financeira do usuário depende de um controle rigoroso nos gastos variáveis (como Groceries e Gas), que são as próximas categorias no ranking.

## Q3 — Análise de Gastos Recorrentes (Contas Fixas)

> Quais despesas se repetem mensalmente?

In [4]:
df_gastos_recorrentes = query("""
SELECT
    Description as Descricao,
    COUNT(DISTINCT strftime('%m', Date)) as Meses_Ativos,
    ROUND(AVG(Amount), 2) as Valor_Medio
FROM transacoes
WHERE \"Transaction Type\" = 'debit'
GROUP BY Description
HAVING Meses_Ativos >= 10
ORDER BY Meses_Ativos DESC;
""")

print(df_gastos_recorrentes.to_string(index=False))

                    Descricao  Meses_Ativos  Valor_Medio
0                  State Farm            12        75.00
1                     Spotify            12        10.69
2               Power Company            12        60.00
3               Phone Company            12        80.02
4                     Netflix            12        12.29
5            Mortgage Payment            12      1178.79
6   Internet Service Provider            12        74.80
7               Grocery Store            12        26.84
8                 Gas Company            12        37.19
9         Credit Card Payment            12       465.37
10         City Water Charges            12        35.00
11                     Amazon            12        33.39
12             Smith and Sons            11       273.89
13               Williams LLC            10       216.72
14                  Starbucks            10         3.91
15                  Smith PLC            10       270.42
16                  Smith Inc  

A economia por 'assinaturas esquecidas' pode ser uma oportunidade imediata de melhoria no saldo mensal.

## Q4 — Perfil de Pagamento

#### Esta query mostra se o usuário usa mais cartão de crédito ou conta corrente, o que ajuda a entender o comportamento de crédito.

> Qual o volume de gastos por cada conta cadastrada?

In [5]:
df_perfil_pagamento = query("""
SELECT
    \"Account Name\" as Conta,
    COUNT(*) as Numero_de_Transacoes,
    ROUND(SUM(Amount), 2) as Total_Gasto
FROM transacoes
WHERE \"Transaction Type\" = 'debit'
GROUP BY \"Account Name\"
ORDER BY Total_Gasto DESC;
""")

total = df_perfil_pagamento['Total_Gasto'].sum()
df_perfil_pagamento['Pct_Gasto'] = (df_perfil_pagamento['Total_Gasto'] / total * 100).round(1)
print(df_perfil_pagamento.to_string(index=False))

         Conta  Numero_de_Transacoes  Total_Gasto
0     checking                 10226   3781214.20
1  credit card                  7502    763621.04



> O cartão é usado para compras de menor valor (conveniência), 
> não para financiar grandes despesas. Comportamento financeiramente saudável na maioria dos usuários.

A análise do Perfil de Pagamento revelou que a conta corrente é o principal veículo de despesa, concentrando mais de 10 mil transações. Isso demonstra um fluxo de caixa de alta rotatividade. O uso do cartão de crédito, embora secundário em volume de transações, representa uma fatia importante do valor total gasto, o que exige atenção à capacidade de endividamento futura do usuário.

## Q5 — Usuários com Saldo Negativo Recorrente

> Quais usuários apresentam saldo negativo em mais de 50% dos meses?

In [6]:
df_neg = query("""
    WITH saldo_mensal AS (
        SELECT
            User_ID,
            strftime('%Y-%m', Date) AS Periodo,
            SUM(CASE WHEN \"Transaction Type\" = 'credit' THEN Amount ELSE -Amount END) AS Saldo
        FROM transacoes
        GROUP BY User_ID, Periodo
    ),
    contagem AS (
        SELECT
            User_ID,
            COUNT(*) AS Total_Meses,
            SUM(CASE WHEN Saldo < 0 THEN 1 ELSE 0 END) AS Meses_Negativos
        FROM saldo_mensal
        GROUP BY User_ID
    )
    SELECT
        User_ID,
        Total_Meses,
        Meses_Negativos,
        ROUND(100.0 * Meses_Negativos / Total_Meses, 1) AS Pct_Negativo
    FROM contagem
    WHERE Pct_Negativo > 50
    ORDER BY Pct_Negativo DESC
""")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(df_neg['User_ID'], df_neg['Pct_Negativo'], color='tomato', edgecolor='white')
ax.axvline(50, color='gray', linestyle='--', linewidth=1, label='Limite 50%')
ax.set_title(f'{len(df_neg)} usuários com saldo negativo em >50% dos meses', fontsize=12)
ax.set_xlabel('% Meses com Saldo Negativo')
ax.legend()
plt.tight_layout()
plt.show()

print(df_neg.to_string(index=False))

      User_ID  Total_Meses  Meses_Negativos  Pct_Negativo
0   USER_0016           24               24         100.0
1   USER_0026           24               24         100.0
2   USER_0031           24               24         100.0
3   USER_0035           24               24         100.0
4   USER_0041           24               24         100.0
5   USER_0007           24               23          95.8
6   USER_0025           24               22          91.7
7   USER_0032           24               22          91.7
8   USER_0017           24               21          87.5
9   USER_0022           24               21          87.5
10  USER_0029           24               21          87.5
11  USER_0038           24               21          87.5
12  USER_0051           24               19          79.2
13  USER_0020           24               18          75.0
14  USER_0003           24               17          70.8


**Resultado Q5 — Usuários com saldo negativo em >50% dos meses:**
- Total identificado: **15 usuários (29%)**
- 5 deles: **100% dos meses no vermelho** (USER_0016, USER_0026, USER_0031, USER_0035, USER_0041)
- Caso mais leve na lista: 70,8% dos meses negativos

> Esses 15 usuários são os candidatos diretos ao perfil "Em Risco Financeiro" descoberto no notebook 04.
> O padrão é consistente.

## Q6 — Taxa de Poupança Média por Usuário

> Taxa = (Receita - Despesa) / Receita

In [7]:
df_poupanca = query("""
    WITH base AS (
        SELECT
            User_ID,
            strftime('%Y-%m', Date) AS Periodo,
            SUM(CASE WHEN \"Transaction Type\" = 'credit' THEN Amount ELSE 0 END) AS Receita,
            SUM(CASE WHEN \"Transaction Type\" = 'debit'  THEN Amount ELSE 0 END) AS Despesa
        FROM transacoes
        GROUP BY User_ID, Periodo
    )
    SELECT
        User_ID,
        ROUND(AVG(CASE WHEN Receita > 0 THEN (Receita - Despesa) / Receita ELSE NULL END), 3) AS Taxa_Poupanca_Media
    FROM base
    GROUP BY User_ID
    ORDER BY Taxa_Poupanca_Media DESC
""")

positivos = (df_poupanca['Taxa_Poupanca_Media'] > 0).sum()
negativos = (df_poupanca['Taxa_Poupanca_Media'] < 0).sum()
print(f'Poupança positiva: {positivos} usuários ({positivos/len(df_poupanca)*100:.0f}%)')
print(f'Poupança negativa: {negativos} usuários ({negativos/len(df_poupanca)*100:.0f}%)')
print(f'Taxa média geral : {df_poupanca["Taxa_Poupanca_Media"].mean()*100:.1f}%')
print()
print(df_poupanca.to_string(index=False))

      User_ID  Taxa_Poupanca_Media
0   USER_0049                0.649
1   USER_0014                0.637
2   USER_0015                0.622
3   USER_0044                0.618
4   USER_0050                0.599
5   USER_0042                0.591
6   USER_0009                0.573
7   USER_0039                0.571
8   USER_0021                0.526
9   USER_0019                0.503
10  USER_0045                0.483
11  USER_0008                0.459
12  USER_0012                0.449
13  USER_0005                0.442
14  USER_0036                0.426
15  USER_0030                0.397
16  USER_0046                0.385
17  USER_0010                0.385
18  USER_0043                0.383
19  USER_0018                0.367
20  USER_0037                0.364
21  USER_0006                0.363
22  USER_0048                0.357
23  USER_0027                0.347
24  USER_0028                0.331
25  USER_0002                0.319
26  USER_0024                0.297
27  USER_0040       



> A distribuição é bimodal: existem dois grupos distintos.
> Não há "quase poupadores", ou o comportamento financeiro é disciplinado ou está em colapso.
> Isso justifica a escolha de clustering com k=3 no notebook 04.

In [9]:
import os
os.makedirs('../sql', exist_ok=True)

queries_content = f"-- Q1: Variação de despesas\n{query_saldo_mensal}\n\n-- Q2: Ranking dos Gastos (Top 10 Categorias)\n{query_top_gastos}\n\n-- Q3: Análise de Gastos Recorrentes\n{query_gastos_recorrentes}\n\n-- Q4: Perfil de Pagamento\n{query_perfil_pagamento}\n\n-- Q5: Usuários com saldo negativo recorrente\n{query_recorrencia_saldo_negativo}\n\n-- Q6: Taxa de Poupança Média por Usuário\n{query_poupanca}\n\n"

with open('../sql/queries_financeiras.sql', 'w') as f:
    f.write(queries_content)

print("Queries exportadas para sql/queries_financeiras.sql! 💾")


Queries exportadas para sql/queries_financeiras.sql! 💾
